## SSH config and the agent

Two things make SSH pleasant to live with: a config file of aliases, and an agent that remembers your unlocked key.

**`~/.ssh/config`** saves per-host settings so you stop typing long command lines:

```
Host work
    HostName 10.0.0.42
    User alice
    IdentityFile ~/.ssh/work_ed25519

Host internal-*
    ProxyJump bastion
```

Now `ssh work` connects as alice to `10.0.0.42` with the right key, and `ssh internal-db1` automatically **jumps through** `bastion`. The rules: each `Host` block matches by pattern (globs allowed), **first match wins**, and a `Host *` block at the end sets defaults. Core keys: **`HostName`**, **`User`**, **`Port`**, **`IdentityFile`**, and **`ProxyJump`** (for bastions).

**`ssh-agent`** holds your decrypted keys in memory so a passphrase-protected key only costs you the passphrase *once* per session:

```bash
ssh-add                 # add ~/.ssh/id_ed25519 to the agent
ssh-add -l              # list keys the agent holds
```

**Agent forwarding** (`ssh -A`, or `ForwardAgent yes`) lets a server you're on use *your* local agent for an onward hop — the key never leaves your laptop. ⚠️ But it lets that intermediate host use your key while you're connected, so **don't forward to untrusted servers**. **`ProxyJump` is usually the safer choice** — it tunnels *through* the bastion without giving it any access to your agent.
